# LeetCode #53 — Maximum Subarray: thuật toán Kadane

**Mức độ:** Medium · **Chủ đề:** Array, Dynamic Programming, Divide and Conquer

Skeleton trên LeetCode:

```python
class Solution:
    def maxSubArray(self, nums: List[int]) -> int:
```

Mục tiêu của notebook là hiểu thật sâu **thuật toán Kadane** — một ví dụ nhập môn tuyệt vời cho **Quy hoạch động (DP)**: định nghĩa đúng "trạng thái" thì lời giải chỉ còn vài dòng.

---

## Sau notebook này, bạn sẽ nắm được

1. Từ `O(n³)` → `O(n²)` (tổng tiền tố / cộng dồn) → `O(n)`.
2. Câu hỏi then chốt của Kadane: **"đoạn tốt nhất kết thúc tại `i` là gì?"**
3. Quyết định ở mỗi bước: **nối tiếp** đoạn cũ hay **bắt đầu lại** từ `nums[i]`.
4. Xử lý mảng **toàn số âm**.
5. Trả về chính đoạn con (không chỉ tổng).

# 1. Bài toán là gì?

Cho mảng số nguyên `nums`. Tìm **đoạn con liên tiếp khác rỗng** có **tổng lớn nhất**, trả về tổng đó.

**Constraints (theo đề):**

- `1 <= nums.length <= 10^5`
- `-10^4 <= nums[i] <= 10^4`

**Follow up:** thử thêm cách **chia để trị (divide and conquer)**.

---

## Ví dụ

```text
[-2, 1, -3, 4, -1, 2, 1, -5, 4]  ->  6      (đoạn [4, -1, 2, 1])
[1]                              ->  1
[5, 4, -1, 7, 8]                 ->  23     (cả mảng)
```

Lưu ý: đoạn con phải **khác rỗng** — với `[-3, -1, -2]`, đáp án là `-1`, không phải `0`.

# 2. Kiến thức nền cần nắm trước

Để hiểu trọn lời giải bài này, bạn cần nắm chắc các kiến thức dưới đây. Mỗi phần trình bày: khái niệm là gì, hình minh họa, code mẫu **chạy được**, độ phức tạp và lỗi hay gặp. Phần nào đã vững, bạn có thể đọc lướt.

1. Quy hoạch động (Dynamic Programming — DP)
2. Chia để trị (divide and conquer)

Cuối mục có phần **2.3 Áp dụng vào bài này** nối các kiến thức trên với lời giải.

## 2.1 Quy hoạch động (Dynamic Programming — DP)

### Khi nào dùng DP?

Quy hoạch động dùng cho bài toán có hai đặc điểm:

1. **Cấu trúc con tối ưu**: đáp án của bài lớn được ghép từ đáp án của các **bài con** nhỏ hơn.
2. **Bài con chồng lấp**: cùng một bài con xuất hiện **nhiều lần** nếu đệ quy thẳng.

Thay vì tính lại, ta **lưu** đáp án mỗi bài con vào bảng và dùng lại → mỗi bài con tính đúng một lần.

### Bốn câu hỏi khi thiết kế DP

1. **Trạng thái**: `dp[i]` nghĩa là gì? (viết thành câu tiếng Việt rõ ràng)
2. **Chuyển trạng thái**: `dp[i]` tính từ những `dp[j]` nào?
3. **Cơ sở**: giá trị nhỏ nhất biết sẵn (`dp[0]`, `dp[1]`…).
4. **Đáp án**: là `dp[n]`, hay `max(dp)`, hay …?

### Ví dụ: leo cầu thang

Mỗi bước leo 1 hoặc 2 bậc. Có bao nhiêu cách leo `n` bậc?

- Trạng thái: `dp[i]` = số cách lên tới bậc `i`.
- Chuyển: bước cuối cùng là 1 bậc (từ `i−1`) hoặc 2 bậc (từ `i−2`) → `dp[i] = dp[i−1] + dp[i−2]`.
- Cơ sở: `dp[0] = 1` (đứng yên), `dp[1] = 1`.

In [1]:
from functools import lru_cache

# Cách 1: đệ quy thuần — chậm vì tính lặp
def leo_de_quy(n):
    return 1 if n <= 1 else leo_de_quy(n - 1) + leo_de_quy(n - 2)

# Cách 2: đệ quy + ghi nhớ (top-down, memoization)
@lru_cache(maxsize=None)
def leo_memo(n):
    return 1 if n <= 1 else leo_memo(n - 1) + leo_memo(n - 2)

# Cách 3: điền bảng từ nhỏ tới lớn (bottom-up, tabulation)
def leo_bang(n):
    dp = [0] * (n + 1)
    dp[0] = 1
    if n >= 1:
        dp[1] = 1
    for i in range(2, n + 1):
        dp[i] = dp[i - 1] + dp[i - 2]
    return dp[n], dp

# Cách 4: chỉ giữ hai giá trị gần nhất — O(1) bộ nhớ
def leo_gon(n):
    a, b = 1, 1                 # dp[i-2], dp[i-1]
    for _ in range(2, n + 1):
        a, b = b, a + b
    return b

print("n = 10:", leo_de_quy(10), leo_memo(10), leo_bang(10)[0], leo_gon(10))
print("bảng dp:", leo_bang(10)[1])

n = 10: 89 89 89 89
bảng dp: [1, 1, 2, 3, 5, 8, 13, 21, 34, 55, 89]


### Top-down hay bottom-up?

| | Top-down (memo) | Bottom-up (bảng) |
|---|---|---|
| Viết | gần với công thức đệ quy, dễ nghĩ | phải chọn **thứ tự** điền bảng |
| Chỉ tính bài con cần thiết | có | không (tính hết) |
| Rủi ro | độ sâu đệ quy | không |
| Tối ưu bộ nhớ | khó | dễ (giữ vài hàng/ô gần nhất) |

### Ví dụ thứ hai: chọn hay không chọn (House Robber)

Dãy nhà có tiền `a[i]`, không được lấy hai nhà **liền nhau**. Lấy tối đa bao nhiêu?

- `dp[i]` = số tiền tối đa khi chỉ xét `i` nhà đầu.
- Nhà thứ `i` (chỉ số `i−1`): **không lấy** → `dp[i−1]`; **lấy** → `dp[i−2] + a[i−1]`. Lấy `max`.

In [2]:
def ke_trom(a):
    dp = [0] * (len(a) + 1)
    if a:
        dp[1] = a[0]
    for i in range(2, len(a) + 1):
        dp[i] = max(dp[i - 1], dp[i - 2] + a[i - 1])
    return dp[-1], dp

print("ke_trom([2, 7, 9, 3, 1]) =", ke_trom([2, 7, 9, 3, 1]))

ke_trom([2, 7, 9, 3, 1]) = (12, [0, 2, 7, 11, 11, 12])


### Lỗi hay gặp

- Định nghĩa trạng thái mơ hồ → công thức sai. Luôn viết `dp[i] = …` bằng lời trước.
- Lệch chỉ số giữa `dp` (dài `n+1`) và mảng đầu vào (dài `n`).
- Điền bảng sai thứ tự (dùng ô chưa tính).
- Quên cơ sở hoặc khởi tạo sai (`0` thay vì `-∞` cho bài tìm max có giá trị âm…).
- Tối ưu bộ nhớ quá sớm khi chưa chắc công thức đúng — hãy viết bản bảng đầy đủ trước.

## 2.2 Chia để trị (divide and conquer)

### Ý tưởng

1. **Chia** bài toán thành các bài con **cùng dạng**, nhỏ hơn (thường là hai nửa).
2. **Trị**: giải các bài con bằng đệ quy (bài đủ nhỏ thì giải trực tiếp).
3. **Kết hợp** kết quả các bài con thành kết quả bài lớn.

Khác DP: các bài con **không chồng lấp** (mỗi phần dữ liệu thuộc đúng một bài con).

```text
            [8 phần tử]                      tầng 0: 1 bài, tổng n việc
         [4]          [4]                    tầng 1: 2 bài, tổng n việc
       [2]  [2]     [2]  [2]                 tầng 2: 4 bài, tổng n việc
      [1][1][1][1] [1][1][1][1]              log₂ n tầng  →  O(n log n)
```

In [3]:
def max_chia_de_tri(a, lo, hi, sau=0):
    """Phần tử lớn nhất của a[lo..hi]."""
    if lo == hi:
        return a[lo]                                   # bài đủ nhỏ
    mid = (lo + hi) // 2
    trai = max_chia_de_tri(a, lo, mid, sau + 1)        # trị nửa trái
    phai = max_chia_de_tri(a, mid + 1, hi, sau + 1)    # trị nửa phải
    kq = max(trai, phai)                               # kết hợp
    print("    " * sau + f"a[{lo}..{hi}] = {a[lo:hi + 1]} → max {kq}")
    return kq

_ = max_chia_de_tri([3, 8, 1, 9, 4, 7, 2, 6], 0, 7)

        a[0..1] = [3, 8] → max 8
        a[2..3] = [1, 9] → max 9
    a[0..3] = [3, 8, 1, 9] → max 9
        a[4..5] = [4, 7] → max 7
        a[6..7] = [2, 6] → max 6
    a[4..7] = [4, 7, 2, 6] → max 7
a[0..7] = [3, 8, 1, 9, 4, 7, 2, 6] → max 9


### Các ví dụ quen thuộc

| Bài | Chia | Kết hợp | Độ phức tạp |
|---|---|---|---|
| Tìm nhị phân | bỏ một nửa | không cần | `O(log n)` |
| Merge sort | hai nửa | trộn `O(n)` | `O(n log n)` |
| Lũy thừa nhanh `x^n` | `x^(n/2)` | bình phương | `O(log n)` |
| Dựng cây cân bằng từ mảng sắp xếp | phần tử giữa làm gốc | gắn hai cây con | `O(n)` |
| Dựng cây từ hai cách duyệt | tách theo vị trí gốc | gắn hai cây con | `O(n)` với hash map |

In [4]:
def luy_thua_nhanh(x, n, sau=0):
    if n == 0:
        return 1
    nua = luy_thua_nhanh(x, n // 2, sau + 1)
    kq = nua * nua * (x if n % 2 else 1)
    print("    " * sau + f"{x}^{n} = ({x}^{n // 2})²{' · ' + str(x) if n % 2 else ''} = {kq}")
    return kq

_ = luy_thua_nhanh(2, 10)

            2^1 = (2^0)² · 2 = 2
        2^2 = (2^1)² = 4
    2^5 = (2^2)² · 2 = 32
2^10 = (2^5)² = 1024


Lỗi hay gặp: chia không đều hoặc không nhỏ đi (`mid` tính sai → đệ quy vô hạn); tạo bản sao `a[:mid]` ở mỗi tầng (tốn thêm `O(n log n)` bộ nhớ — truyền chỉ số `lo, hi` thay vì cắt list); quên trường hợp đoạn rỗng.

## 2.3 Áp dụng vào bài này

**Kadane** là quy hoạch động: `best_end[i]` = tổng lớn nhất của đoạn con **kết thúc tại** `i` = `max(a[i], best_end[i−1] + a[i])`; đáp án là max qua mọi `i`, và chỉ cần giữ một biến. Cách **chia để trị**: đoạn tốt nhất nằm hẳn bên trái, hẳn bên phải, hoặc **cắt qua giữa**.

# 3. Cách 1: thử mọi đoạn — `O(n³)`

In [5]:
from typing import List

def max_subarray_cubic(nums: List[int]) -> int:
    n = len(nums)
    best = nums[0]
    for i in range(n):
        for j in range(i, n):
            best = max(best, sum(nums[i:j + 1]))
    return best

print(max_subarray_cubic([-2, 1, -3, 4, -1, 2, 1, -5, 4]))

6


# 4. Cách 2: cộng dồn — `O(n²)`

`sum(nums[i:j+1])` tính lại từ đầu mỗi lần. Thay vào đó, với `i` cố định, **cộng dồn** khi `j` tăng:

In [6]:
def max_subarray_quadratic(nums: List[int]) -> int:
    n = len(nums)
    best = nums[0]
    for i in range(n):
        running = 0
        for j in range(i, n):
            running += nums[j]
            best = max(best, running)
    return best

print(max_subarray_quadratic([-2, 1, -3, 4, -1, 2, 1, -5, 4]))

6


Với `n = 10^5`, `n² / 2 = 5 * 10^9` — vẫn quá chậm.

# 5. Ý tưởng Kadane: đoạn tốt nhất **kết thúc tại `i`**

Thay vì hỏi "đoạn tốt nhất là gì?", hỏi một câu **nhỏ hơn**:

> `cur[i]` = tổng lớn nhất của một đoạn **kết thúc đúng tại vị trí `i`**.

Đáp án cuối = `max(cur[0], cur[1], ..., cur[n-1])` (vì đoạn tốt nhất phải kết thúc ở đâu đó).

## Tính `cur[i]` từ `cur[i-1]`

Một đoạn kết thúc tại `i` chỉ có **hai khả năng**:

1. **Chỉ gồm `nums[i]`** — bắt đầu lại từ đây.
2. **Nối `nums[i]` vào sau** một đoạn kết thúc tại `i - 1` — tốt nhất là đoạn có tổng `cur[i-1]`.

Vậy:

```text
cur[i] = max(nums[i], cur[i-1] + nums[i])
```

Trực giác: nếu `cur[i-1] < 0`, đoạn phía trước chỉ **kéo tổng xuống** → bỏ nó, bắt đầu lại. Nếu `cur[i-1] >= 0`, nối tiếp luôn có lợi (hoặc không hại).

In [7]:
nums = [-2, 1, -3, 4, -1, 2, 1, -5, 4]
cur = [0] * len(nums)
cur[0] = nums[0]
for i in range(1, len(nums)):
    cur[i] = max(nums[i], cur[i - 1] + nums[i])
print("nums:", nums)
print("cur :", cur)
print("max(cur) =", max(cur))

nums: [-2, 1, -3, 4, -1, 2, 1, -5, 4]
cur : [-2, 1, -2, 4, 3, 5, 6, 1, 5]
max(cur) = 6


## Chỉ cần nhớ giá trị trước

`cur[i]` chỉ phụ thuộc `cur[i-1]` → không cần cả mảng, một biến là đủ. Bộ nhớ `O(1)`.

In [8]:
def max_subarray(nums: List[int]) -> int:
    cur = best = nums[0]
    for x in nums[1:]:
        cur = max(x, cur + x)        # nối tiếp hay bắt đầu lại?
        best = max(best, cur)
    return best

print(max_subarray([-2, 1, -3, 4, -1, 2, 1, -5, 4]))
print(max_subarray([1]))
print(max_subarray([5, 4, -1, 7, 8]))
print(max_subarray([-3, -1, -2]))

6
1
23
-1


# 6. Mô phỏng từng bước

`nums = [-2, 1, -3, 4, -1, 2, 1, -5, 4]`

| i | x | cur trước | cur + x | max(x, cur + x) | Quyết định | best |
|---|---|---|---|---|---|---|
| 0 | -2 | — | — | -2 | khởi tạo | -2 |
| 1 | 1 | -2 | -1 | 1 | **bắt đầu lại** | 1 |
| 2 | -3 | 1 | -2 | -2 | nối tiếp | 1 |
| 3 | 4 | -2 | 2 | 4 | **bắt đầu lại** | 4 |
| 4 | -1 | 4 | 3 | 3 | nối tiếp | 4 |
| 5 | 2 | 3 | 5 | 5 | nối tiếp | 5 |
| 6 | 1 | 5 | 6 | 6 | nối tiếp | **6** |
| 7 | -5 | 6 | 1 | 1 | nối tiếp | 6 |
| 8 | 4 | 1 | 5 | 5 | nối tiếp | 6 |

In [9]:
def trace_kadane(nums: List[int]) -> int:
    cur = best = nums[0]
    start = best_start = best_end = 0
    print(f"i=0 x={nums[0]:>3} | khởi tạo cur = best = {cur}")
    for i in range(1, len(nums)):
        x = nums[i]
        if cur + x < x:
            cur = x
            start = i
            decision = "bắt đầu lại"
        else:
            cur = cur + x
            decision = "nối tiếp   "
        if cur > best:
            best, best_start, best_end = cur, start, i
        print(f"i={i} x={x:>3} | {decision} -> cur = {cur:>3} (đoạn {nums[start:i + 1]}) | best = {best}")
    print(f"Đáp án {best}, đoạn {nums[best_start:best_end + 1]}")
    return best

_ = trace_kadane([-2, 1, -3, 4, -1, 2, 1, -5, 4])

i=0 x= -2 | khởi tạo cur = best = -2
i=1 x=  1 | bắt đầu lại -> cur =   1 (đoạn [1]) | best = 1
i=2 x= -3 | nối tiếp    -> cur =  -2 (đoạn [1, -3]) | best = 1
i=3 x=  4 | bắt đầu lại -> cur =   4 (đoạn [4]) | best = 4
i=4 x= -1 | nối tiếp    -> cur =   3 (đoạn [4, -1]) | best = 4
i=5 x=  2 | nối tiếp    -> cur =   5 (đoạn [4, -1, 2]) | best = 5
i=6 x=  1 | nối tiếp    -> cur =   6 (đoạn [4, -1, 2, 1]) | best = 6
i=7 x= -5 | nối tiếp    -> cur =   1 (đoạn [4, -1, 2, 1, -5]) | best = 6
i=8 x=  4 | nối tiếp    -> cur =   5 (đoạn [4, -1, 2, 1, -5, 4]) | best = 6
Đáp án 6, đoạn [4, -1, 2, 1]


# 7. Cách 3 (follow up): Chia để trị

Chia mảng làm đôi tại `mid`. Đoạn tốt nhất hoặc:

1. nằm hẳn ở **nửa trái**,
2. nằm hẳn ở **nửa phải**,
3. **cắt qua giữa**: gồm một phần đuôi của nửa trái (kết thúc tại `mid`) + một phần đầu của nửa phải (bắt đầu tại `mid + 1`).

Trường hợp 3 tính được trong `O(n)` bằng cách đi từ `mid` ra hai phía. Tổng thời gian `T(n) = 2T(n/2) + O(n) = O(n log n)` — chậm hơn Kadane, nhưng là bài tập tốt về chia để trị.

In [10]:
def max_subarray_dc(nums: List[int]) -> int:
    def solve(lo: int, hi: int) -> int:
        if lo == hi:
            return nums[lo]
        mid = (lo + hi) // 2
        left_best = solve(lo, mid)
        right_best = solve(mid + 1, hi)

        s, left_tail = 0, float("-inf")
        for i in range(mid, lo - 1, -1):
            s += nums[i]
            left_tail = max(left_tail, s)
        s, right_head = 0, float("-inf")
        for i in range(mid + 1, hi + 1):
            s += nums[i]
            right_head = max(right_head, s)

        return max(left_best, right_best, left_tail + right_head)

    return solve(0, len(nums) - 1)

print(max_subarray_dc([-2, 1, -3, 4, -1, 2, 1, -5, 4]))

6


# 8. Phiên bản đúng format LeetCode

In [11]:
class Solution:
    def maxSubArray(self, nums: List[int]) -> int:
        cur = best = nums[0]
        for x in nums[1:]:
            cur = max(x, cur + x)
            best = max(best, cur)
        return best

In [12]:
def check(name, actual, expected):
    assert actual == expected, (name, actual, expected)
    print(f"{name:<24} PASS  {actual}")


sol = Solution()
check("Ví dụ 1", sol.maxSubArray([-2, 1, -3, 4, -1, 2, 1, -5, 4]), 6)
check("Ví dụ 2", sol.maxSubArray([1]), 1)
check("Ví dụ 3", sol.maxSubArray([5, 4, -1, 7, 8]), 23)
check("Toàn số âm", sol.maxSubArray([-3, -1, -2]), -1)
check("Một số âm", sol.maxSubArray([-5]), -5)
check("Có số 0", sol.maxSubArray([0, -1, 0]), 0)
check("Đoạn ở cuối", sol.maxSubArray([-1, -2, 3, 4]), 7)

import random
random.seed(53)
for _ in range(3000):
    nums = [random.randint(-10, 10) for _ in range(random.randint(1, 15))]
    expected = max_subarray_cubic(nums)
    assert sol.maxSubArray(nums) == expected, nums
    assert max_subarray_quadratic(nums) == expected
    assert max_subarray_dc(nums) == expected
print("3000 test ngẫu nhiên: Kadane, O(n²), chia để trị đều khớp O(n³)")

Ví dụ 1                  PASS  6
Ví dụ 2                  PASS  1
Ví dụ 3                  PASS  23
Toàn số âm               PASS  -1
Một số âm                PASS  -5
Có số 0                  PASS  0
Đoạn ở cuối              PASS  7


3000 test ngẫu nhiên: Kadane, O(n²), chia để trị đều khớp O(n³)


# 9. Vì sao đúng? Độ phức tạp?

**Bất biến:** sau khi xử lý `nums[i]`:

- `cur` = tổng lớn nhất của đoạn **kết thúc tại `i`**,
- `best` = tổng lớn nhất của mọi đoạn **kết thúc tại một vị trí `<= i`**.

`cur` đúng nhờ lập luận "chỉ có hai khả năng" (mục 5). `best` đúng vì nó là max của các `cur`.

Khi duyệt xong, `best` xét mọi vị trí kết thúc → là đáp án.

```text
Time:  O(n)
Space: O(1)
```

| Cách | Thời gian | Bộ nhớ |
|---|---|---|
| Thử mọi đoạn + sum | O(n³) | O(1) |
| Cộng dồn | O(n²) | O(1) |
| Chia để trị | O(n log n) | O(log n) |
| Kadane | O(n) | O(1) |

# 10. Những lỗi dễ mắc

## Lỗi 1: Khởi tạo `best = 0`

Với mảng toàn số âm, trả về `0` — tương ứng đoạn **rỗng**, mà đề yêu cầu khác rỗng.

In [13]:
def wrong_init_zero(nums: List[int]) -> int:
    cur = best = 0                          # SAI
    for x in nums:
        cur = max(x, cur + x)
        best = max(best, cur)
    return best

print("Sai :", wrong_init_zero([-3, -1, -2]), "| Đúng:", sol.maxSubArray([-3, -1, -2]))

Sai : 0 | Đúng: -1


## Lỗi 2: Phiên bản "reset về 0" viết sai thứ tự

Một cách viết Kadane phổ biến: `cur += x; best = max(best, cur); if cur < 0: cur = 0`. Nếu đặt `if cur < 0: cur = 0` **trước** khi cập nhật `best`, mảng toàn âm lại trả về `0`.

In [14]:
def wrong_reset_order(nums: List[int]) -> int:
    cur, best = 0, nums[0]
    for x in nums:
        cur += x
        if cur < 0:
            cur = 0                          # SAI: reset trước khi cập nhật best
        best = max(best, cur)
    return best

def kadane_reset_ok(nums: List[int]) -> int:
    cur, best = 0, nums[0]
    for x in nums:
        cur += x
        best = max(best, cur)                # cập nhật trước
        if cur < 0:
            cur = 0
    return best

print("Sai :", wrong_reset_order([-3, -1, -2]), "| Đúng:", kadane_reset_ok([-3, -1, -2]))

Sai : 0 | Đúng: -1


## Lỗi 3: Nhầm với "dãy con" (không liên tiếp)

Nếu được chọn các phần tử **không liên tiếp**, đáp án đơn giản là tổng mọi số dương (hoặc số lớn nhất nếu toàn âm). Bài này yêu cầu **liên tiếp**.

## Lỗi 4: Dùng Sliding Window

Kadane **trông giống** sliding window, nhưng tổng đoạn **không đơn điệu** khi có số âm — không thể dùng quy tắc "thu hẹp khi vi phạm" của bài #3. Kadane là DP, không phải two pointers.

# 11. Mẹo nhận diện pattern

**"Tốt nhất kết thúc tại `i`"** là mẫu DP trên mảng cực kỳ phổ biến:

| Bài | Trạng thái `dp[i]` |
|---|---|
| #53 Maximum Subarray | tổng lớn nhất của đoạn kết thúc tại `i` |
| #152 Maximum Product Subarray | **cả** tích lớn nhất **và** nhỏ nhất kết thúc tại `i` (số âm đảo dấu!) |
| #918 Maximum Sum Circular Subarray | Kadane max + Kadane min |
| #121 Best Time to Buy and Sell Stock | lãi lớn nhất nếu bán tại `i` |
| #300 Longest Increasing Subsequence | độ dài LIS kết thúc tại `i` |

Câu hỏi để tìm trạng thái: *"Nếu biết đáp án cho mọi tiền tố kết thúc tại `i - 1`, mình mở rộng tới `i` thế nào?"*

## Template

```python
cur = best = nums[0]
for x in nums[1:]:
    cur = max(x, cur + x)
    best = max(best, cur)
return best
```

# 12. Bài tập vận dụng

## Bài 1 — Tính bằng tay

Lập bảng Kadane cho `[2, -3, 4, -1, -2, 1, 5, -3]`. Đáp án là bao nhiêu, đoạn nào?

## Bài 2 — Điền chỗ trống

In [15]:
def exercise_2_max_subarray(nums: List[int]) -> int:
    cur = best = nums[0]
    for x in nums[1:]:
        # TODO 1: cập nhật cur: nối tiếp hay bắt đầu lại?
        # TODO 2: cập nhật best
        pass
    return best


# assert exercise_2_max_subarray([-2, 1, -3, 4, -1, 2, 1, -5, 4]) == 6
# assert exercise_2_max_subarray([-3, -1, -2]) == -1

## Bài 3 — Biến thể: trả về chỉ số đoạn con

Trả về `(tổng, start, end)` của đoạn tổng lớn nhất (nếu nhiều đoạn bằng nhau, lấy đoạn **kết thúc sớm nhất**, và trong các đoạn đó lấy đoạn **ngắn nhất**).

```python
max_subarray_range([-2, 1, -3, 4, -1, 2, 1, -5, 4]) -> (6, 3, 6)
max_subarray_range([-3, -1, -2])                    -> (-1, 1, 1)
```

In [16]:
def exercise_3_max_subarray_range(nums: List[int]):
    # TODO
    pass


# assert exercise_3_max_subarray_range([-2, 1, -3, 4, -1, 2, 1, -5, 4]) == (6, 3, 6)
# assert exercise_3_max_subarray_range([-3, -1, -2]) == (-1, 1, 1)
# assert exercise_3_max_subarray_range([5]) == (5, 0, 0)

## Bài 4 — Biến thể: Maximum Product Subarray (LeetCode #152)

Tìm **tích** lớn nhất của một đoạn con liên tiếp.

```python
max_product([2, 3, -2, 4])  -> 6
max_product([-2, 0, -1])    -> 0
max_product([-2, 3, -4])    -> 24
```

Gợi ý nhanh: số âm × số âm = số dương, nên phải giữ **cả** tích lớn nhất và tích nhỏ nhất kết thúc tại `i`.

In [17]:
def exercise_4_max_product(nums: List[int]) -> int:
    # TODO
    pass


# assert exercise_4_max_product([2, 3, -2, 4]) == 6
# assert exercise_4_max_product([-2, 0, -1]) == 0
# assert exercise_4_max_product([-2, 3, -4]) == 24

## Bài 5 — Mini challenge: Maximum Sum Circular Subarray (LeetCode #918)

Mảng **vòng tròn**: đoạn con có thể "quấn" từ cuối về đầu (mỗi phần tử dùng tối đa một lần).

```python
max_circular([1, -2, 3, -2])   -> 3
max_circular([5, -3, 5])       -> 10     (5 + 5, quấn vòng)
max_circular([-3, -2, -3])     -> -2
```

Gợi ý nhanh: đoạn quấn vòng = **tổng cả mảng − đoạn tổng nhỏ nhất** (ở giữa). Cẩn thận khi mọi số đều âm.

In [18]:
def exercise_5_max_circular(nums: List[int]) -> int:
    # TODO
    pass


# assert exercise_5_max_circular([1, -2, 3, -2]) == 3
# assert exercise_5_max_circular([5, -3, 5]) == 10
# assert exercise_5_max_circular([-3, -2, -3]) == -2

# 13. Gợi ý

**Bài 1**
- Mức 1: dùng `trace_kadane` để kiểm tra lại sau khi tự lập bảng.

**Bài 2**
- Mức 1: `cur = max(x, cur + x)`.

**Bài 3**
- Mức 1: giữ `start` của đoạn hiện tại; khi "bắt đầu lại" (tức `x > cur + x`, nghĩa là `cur < 0`), đặt `start = i`.
- Mức 2: chỉ cập nhật `best` khi `cur > best` (lớn hơn **hẳn**) để giữ đoạn kết thúc sớm nhất.
- Mức 3: khi `cur + x == x` (tức `cur == 0`), **bắt đầu lại** để có đoạn ngắn hơn.

**Bài 4**
- Mức 1: `hi, lo = max(x, hi*x, lo*x), min(x, hi*x, lo*x)` (tính đồng thời từ giá trị cũ).

**Bài 5**
- Mức 1: `total = sum(nums)`, `max_k` = Kadane max, `min_k` = Kadane min.
- Mức 2: đáp án `max(max_k, total - min_k)`, trừ khi `max_k < 0` (mọi số âm) → trả `max_k`.

# 14. Đáp án tham khảo

Bạn nên tự làm trước khi xem phần này.

---

## Đáp án Bài 1

In [19]:
_ = trace_kadane([2, -3, 4, -1, -2, 1, 5, -3])

i=0 x=  2 | khởi tạo cur = best = 2
i=1 x= -3 | nối tiếp    -> cur =  -1 (đoạn [2, -3]) | best = 2
i=2 x=  4 | bắt đầu lại -> cur =   4 (đoạn [4]) | best = 4
i=3 x= -1 | nối tiếp    -> cur =   3 (đoạn [4, -1]) | best = 4
i=4 x= -2 | nối tiếp    -> cur =   1 (đoạn [4, -1, -2]) | best = 4
i=5 x=  1 | nối tiếp    -> cur =   2 (đoạn [4, -1, -2, 1]) | best = 4
i=6 x=  5 | nối tiếp    -> cur =   7 (đoạn [4, -1, -2, 1, 5]) | best = 7
i=7 x= -3 | nối tiếp    -> cur =   4 (đoạn [4, -1, -2, 1, 5, -3]) | best = 7
Đáp án 7, đoạn [4, -1, -2, 1, 5]


## Đáp án Bài 2

In [20]:
def answer_2_max_subarray(nums: List[int]) -> int:
    cur = best = nums[0]
    for x in nums[1:]:
        cur = max(x, cur + x)
        best = max(best, cur)
    return best

assert answer_2_max_subarray([-2, 1, -3, 4, -1, 2, 1, -5, 4]) == 6
assert answer_2_max_subarray([-3, -1, -2]) == -1
print("Bài 2 PASS")

Bài 2 PASS


## Đáp án Bài 3

In [21]:
def answer_3_max_subarray_range(nums: List[int]):
    cur = best = nums[0]
    start = best_start = best_end = 0
    for i in range(1, len(nums)):
        x = nums[i]
        if cur <= 0:                 # bắt đầu lại (cả khi cur == 0 để đoạn ngắn hơn)
            cur, start = x, i
        else:
            cur += x
        if cur > best:
            best, best_start, best_end = cur, start, i
    return best, best_start, best_end

assert answer_3_max_subarray_range([-2, 1, -3, 4, -1, 2, 1, -5, 4]) == (6, 3, 6)
assert answer_3_max_subarray_range([-3, -1, -2]) == (-1, 1, 1)
assert answer_3_max_subarray_range([5]) == (5, 0, 0)
for _ in range(2000):
    nums = [random.randint(-6, 6) for _ in range(random.randint(1, 10))]
    s, a, b = answer_3_max_subarray_range(nums)
    assert s == sum(nums[a:b + 1]) == sol.maxSubArray(nums)
print("Bài 3 PASS")

Bài 3 PASS


## Đáp án Bài 4

In [22]:
def answer_4_max_product(nums: List[int]) -> int:
    hi = lo = best = nums[0]
    for x in nums[1:]:
        hi, lo = max(x, hi * x, lo * x), min(x, hi * x, lo * x)
        best = max(best, hi)
    return best

assert answer_4_max_product([2, 3, -2, 4]) == 6
assert answer_4_max_product([-2, 0, -1]) == 0
assert answer_4_max_product([-2, 3, -4]) == 24
print("Bài 4 PASS")

Bài 4 PASS


Phải tính `hi` và `lo` mới **đồng thời** từ giá trị cũ (gán đồng thời bằng tuple), nếu không `lo` sẽ dùng `hi` đã bị cập nhật.

## Đáp án Bài 5

In [23]:
def answer_5_max_circular(nums: List[int]) -> int:
    total = 0
    cur_max = cur_min = 0
    max_k, min_k = nums[0], nums[0]
    for x in nums:
        cur_max = max(x, cur_max + x)
        max_k = max(max_k, cur_max)
        cur_min = min(x, cur_min + x)
        min_k = min(min_k, cur_min)
        total += x
    if max_k < 0:
        return max_k
    return max(max_k, total - min_k)

assert answer_5_max_circular([1, -2, 3, -2]) == 3
assert answer_5_max_circular([5, -3, 5]) == 10
assert answer_5_max_circular([-3, -2, -3]) == -2

def circular_brute(nums):
    n, best = len(nums), nums[0]
    for i in range(n):
        s = 0
        for length in range(1, n + 1):
            s += nums[(i + length - 1) % n]
            best = max(best, s)
    return best

for _ in range(2000):
    nums = [random.randint(-6, 6) for _ in range(random.randint(1, 9))]
    assert answer_5_max_circular(nums) == circular_brute(nums), nums
print("Bài 5 PASS (khớp brute force vòng tròn trên 2000 test)")

Bài 5 PASS (khớp brute force vòng tròn trên 2000 test)


# 15. Tổng kết

> Hỏi "đoạn tốt nhất **kết thúc tại `i`**". Mỗi bước chỉ có hai lựa chọn: **nối tiếp** (`cur + x`) hoặc **bắt đầu lại** (`x`). Lấy max của mọi `cur` là đáp án.

---

## Checklist trước khi nộp

- [ ] Khởi tạo `cur = best = nums[0]` (không phải 0)?
- [ ] `cur = max(x, cur + x)`?
- [ ] `best = max(best, cur)` sau mỗi bước?
- [ ] Đã thử mảng toàn số âm, mảng 1 phần tử?